In [1]:
# Uncomment if dependencies are missing:
# %pip install torch torchvision transformers albumentations opencv-python evaluate tqdm matplotlib pandas Pillow

In [2]:
import os

# Set before importing torch (restart kernel after changing).
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
import base64
import io
import json
import random
import zlib
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

# --- Hyperparameters ---
IMAGE_SIZE = 512
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 2  # >1 can increase peak VRAM with Mask2Former; raise only if stable
NUM_EPOCHS = 10
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
NUM_WORKERS = 0  # Windows/Jupyter-safe
USE_AMP = True
# HF forward still needs intermediate decoder states; we skip auxiliary *loss* separately.
SKIP_AUXILIARY_LOSS = True
TRAIN_NUM_POINTS = 4096  # default 12544 — major VRAM saver during loss
USE_GRAD_CHECKPOINTING = True
EMPTY_CACHE_EVERY = 25
VAL_COMPUTE_MIOU = False  # val loss only each epoch; set True if you have headroom
IOU_INTERVAL = 100  # used when VAL_COMPUTE_MIOU=True
PROGRESS_IOU_INTERVAL = 50  # mIoU in tqdm bar every N batches (0=disable)
SEED = 42

PRETRAINED_MODEL = "facebook/mask2former-swin-small-ade-semantic"
LOAD_CHECKPOINT: Optional[str] = None  # e.g. "checkpoints/mask2former/best"

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# --- Paths ---
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "dataset"
META_PATH = DATASET_ROOT / "meta.json"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "mask2former"

if not META_PATH.exists():
    PROJECT_ROOT = PROJECT_ROOT / "HSS Project"
    DATASET_ROOT = PROJECT_ROOT / "dataset"
    META_PATH = DATASET_ROOT / "meta.json"
    CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "mask2former"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

classes = meta["classes"]
num_classes = len(classes) + 1  # 103 food classes + background

class_id_to_train_id: Dict[int, int] = {}
train_id_to_class_name: Dict[int, str] = {0: "background"}
train_id_to_color: Dict[int, Tuple[int, int, int]] = {0: (0, 0, 0)}

for idx, cls in enumerate(classes, start=1):
    class_id_to_train_id[cls["id"]] = idx
    train_id_to_class_name[idx] = cls["title"]
    color_hex = cls.get("color", "#FFFFFF").lstrip("#")
    train_id_to_color[idx] = tuple(int(color_hex[i : i + 2], 16) for i in (0, 2, 4))

id2label = {i: train_id_to_class_name[i] for i in range(num_classes)}
label2id = {v: k for k, v in id2label.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Dataset root: {DATASET_ROOT}")
print(f"Classes: {len(classes)} (+ background) = {num_classes}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"device: {device}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

Dataset root: u:\Dev\AI\HSS Project\dataset
Classes: 103 (+ background) = 104
Checkpoint dir: u:\Dev\AI\HSS Project\checkpoints\mask2former
device: cuda:0
Effective batch size: 8


In [3]:
def decode_bitmap_to_canvas(bitmap_data: str, origin: List[int], canvas_h: int, canvas_w: int) -> np.ndarray:
    """Decode Supervisely-style bitmap into a full-size binary mask."""
    packed_bytes = base64.b64decode(bitmap_data)
    try:
        png_bytes = zlib.decompress(packed_bytes)
    except zlib.error:
        png_bytes = packed_bytes

    bitmap_arr = np.array(Image.open(io.BytesIO(png_bytes)).convert("RGBA"))
    local_mask = bitmap_arr[..., 3] > 0

    x0, y0 = int(origin[0]), int(origin[1])
    h, w = local_mask.shape
    canvas = np.zeros((canvas_h, canvas_w), dtype=bool)

    x1, y1 = max(0, x0), max(0, y0)
    x2, y2 = min(canvas_w, x0 + w), min(canvas_h, y0 + h)
    if x1 >= x2 or y1 >= y2:
        return canvas

    src_x1, src_y1 = x1 - x0, y1 - y0
    src_x2, src_y2 = src_x1 + (x2 - x1), src_y1 + (y2 - y1)
    canvas[y1:y2, x1:x2] = local_mask[src_y1:src_y2, src_x1:src_x2]
    return canvas


class FoodSegRawDataset(Dataset):
    """Load native-resolution RGB image and semantic mask from Supervisely JSON."""

    def __init__(
        self,
        dataset_root: Path,
        split: str = "train",
        class_id_to_train_id: Optional[Dict[int, int]] = None,
    ):
        assert split in {"train", "test"}
        self.dataset_root = Path(dataset_root)
        self.split = split
        self.img_dir = self.dataset_root / split / "img"
        self.ann_dir = self.dataset_root / split / "ann"
        self.class_id_to_train_id = class_id_to_train_id or {}

        self.samples = []
        for ann_path in sorted(self.ann_dir.glob("*.json")):
            img_name = ann_path.name.replace(".json", "")
            img_path = self.img_dir / img_name
            if img_path.exists():
                self.samples.append((img_path, ann_path))

        if not self.samples:
            raise RuntimeError(f"No samples found for split={split} in {self.dataset_root}")

    def __len__(self) -> int:
        return len(self.samples)

    def _build_mask_from_annotation(self, ann: dict, canvas_h: int, canvas_w: int) -> np.ndarray:
        sem_mask = np.zeros((canvas_h, canvas_w), dtype=np.int64)

        for obj in ann.get("objects", []):
            if obj.get("geometryType") != "bitmap":
                continue
            bitmap = obj.get("bitmap")
            if bitmap is None:
                continue
            class_id = obj.get("classId")
            train_id = self.class_id_to_train_id.get(class_id, 0)
            obj_mask = decode_bitmap_to_canvas(
                bitmap_data=bitmap["data"],
                origin=bitmap["origin"],
                canvas_h=canvas_h,
                canvas_w=canvas_w,
            )
            sem_mask[obj_mask] = train_id
        return sem_mask

    @staticmethod
    def _align_mask_to_image(mask: np.ndarray, image: np.ndarray, ann: dict) -> np.ndarray:
        """Handle rare FoodSeg103 samples where JSON size disagrees with the JPEG."""
        h_img, w_img = image.shape[:2]
        if mask.shape == (h_img, w_img):
            return mask

        ann_h = int(ann["size"]["height"])
        ann_w = int(ann["size"]["width"])
        if mask.shape == (ann_h, ann_w) and (ann_h, ann_w) == (w_img, h_img):
            return np.transpose(mask, (1, 0))

        return cv2.resize(
            mask.astype(np.uint8),
            (w_img, h_img),
            interpolation=cv2.INTER_NEAREST,
        ).astype(np.int64)

    def __getitem__(self, idx: int):
        img_path, ann_path = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"), dtype=np.uint8)
        with open(ann_path, "r", encoding="utf-8") as f:
            ann = json.load(f)
        ann_h = int(ann["size"]["height"])
        ann_w = int(ann["size"]["width"])
        mask = self._build_mask_from_annotation(ann, canvas_h=ann_h, canvas_w=ann_w)
        mask = self._align_mask_to_image(mask, image, ann)
        return image, mask, {"image_path": str(img_path), "index": idx}

In [4]:
import albumentations as A
import cv2


def _build_geom_color_transforms(image_size: int, train: bool) -> A.Compose:
    resize_pad = [
        A.LongestMaxSize(max_size=image_size, interpolation=cv2.INTER_LINEAR),
        A.PadIfNeeded(
            min_height=image_size,
            min_width=image_size,
            border_mode=cv2.BORDER_CONSTANT,
            fill=0,
            fill_mask=0,
        ),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]

    if not train:
        return A.Compose(resize_pad, additional_targets={"mask": "mask"})

    # GaussNoise API differs across albumentations versions.
    try:
        noise = A.GaussNoise(std_range=(0.02, 0.06), p=1.0)
    except TypeError:
        noise = A.GaussNoise(var_limit=(10.0, 50.0), p=1.0)

    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.1,
                rotate_limit=15,
                border_mode=cv2.BORDER_CONSTANT,
                fill=0,
                fill_mask=0,
                p=0.5,
            ),
            A.RandomBrightnessContrast(p=0.4),
            A.HueSaturationValue(p=0.3),
            A.OneOf([noise, A.GaussianBlur(blur_limit=(3, 5), p=1.0)], p=0.2),
            *resize_pad,
        ],
        additional_targets={"mask": "mask"},
    )


class FoodSegMask2FormerDataset(Dataset):
    """Wrap raw dataset with Albumentations; keep originals for eval."""

    def __init__(self, raw_dataset: FoodSegRawDataset, transform: A.Compose):
        self.raw_dataset = raw_dataset
        self.transform = transform

    def __len__(self) -> int:
        return len(self.raw_dataset)

    def __getitem__(self, idx: int):
        image, mask, info = self.raw_dataset[idx]
        orig_image = image.copy()
        orig_mask = mask.copy()

        out = self.transform(image=image, mask=mask)
        aug_image = out["image"]
        aug_mask = out["mask"].astype(np.int64)

        return aug_image, aug_mask, orig_image, orig_mask


train_raw = FoodSegRawDataset(DATASET_ROOT, split="train", class_id_to_train_id=class_id_to_train_id)
test_raw = FoodSegRawDataset(DATASET_ROOT, split="test", class_id_to_train_id=class_id_to_train_id)

train_transform = _build_geom_color_transforms(IMAGE_SIZE, train=True)
test_transform = _build_geom_color_transforms(IMAGE_SIZE, train=False)

train_ds = FoodSegMask2FormerDataset(train_raw, train_transform)
test_ds = FoodSegMask2FormerDataset(test_raw, test_transform)

print(f"train samples: {len(train_ds)}")
print(f"test samples:  {len(test_ds)}")

train samples: 4983
test samples:  2135


u:\Programms\Python313\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
from transformers import Mask2FormerImageProcessor

processor = Mask2FormerImageProcessor(
    ignore_index=0,
    reduce_labels=False,
    do_resize=False,
    do_rescale=False,
    do_normalize=False,
)


def collate_fn(batch):
    images, seg_maps, orig_images, orig_seg_maps = zip(*batch)
    batch_out = processor(list(images), segmentation_maps=list(seg_maps), return_tensors="pt")
    batch_out["original_images"] = list(orig_images)
    batch_out["original_segmentation_maps"] = list(orig_seg_maps)
    return batch_out


train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
test_dl = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)

# Sanity check one sample
aug_img, aug_mask, orig_img, orig_mask = train_ds[0]
print("aug image:", aug_img.shape, aug_img.dtype, f"[{aug_img.min():.2f}, {aug_img.max():.2f}]")
print("aug mask:", aug_mask.shape, "unique classes:", np.unique(aug_mask)[:10])
print("orig image:", orig_img.shape, "orig mask:", orig_mask.shape)

aug image: (512, 512, 3) float32 [-2.12, 2.59]
aug mask: (512, 512) unique classes: [ 0 22 81 89]
orig image: (384, 512, 3) orig mask: (384, 512)


In [6]:
from transformers import Mask2FormerConfig, Mask2FormerForUniversalSegmentation
import types


def get_model(load_checkpoint: Optional[str] = None):
    if load_checkpoint:
        print(f"Loading checkpoint from {load_checkpoint}")
        model = Mask2FormerForUniversalSegmentation.from_pretrained(load_checkpoint)
    else:
        print(f"Loading pretrained weights from {PRETRAINED_MODEL} ...")
        config = Mask2FormerConfig.from_pretrained(PRETRAINED_MODEL)
        config.num_labels = num_classes
        config.id2label = id2label
        config.label2id = label2id
        config.train_num_points = TRAIN_NUM_POINTS
        config.use_auxiliary_loss = True  # required: HF forward iterates decoder states

        model = Mask2FormerForUniversalSegmentation.from_pretrained(
            PRETRAINED_MODEL,
            config=config,
            ignore_mismatched_sizes=True,
        )
    return model


def patch_skip_auxiliary_loss(model):
    if not SKIP_AUXILIARY_LOSS:
        return

    def get_loss_dict_no_aux(
        self,
        masks_queries_logits,
        class_queries_logits,
        mask_labels,
        class_labels,
        auxiliary_predictions,
    ):
        return Mask2FormerForUniversalSegmentation.get_loss_dict(
            self,
            masks_queries_logits,
            class_queries_logits,
            mask_labels,
            class_labels,
            auxiliary_predictions=[],
        )

    model.get_loss_dict = types.MethodType(get_loss_dict_no_aux, model)


model = get_model(LOAD_CHECKPOINT)
patch_skip_auxiliary_loss(model)

if USE_GRAD_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled.")
    except ValueError as exc:
        print(f"Gradient checkpointing unavailable: {exc}")

model.to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.1, patience=3, threshold=1e-4
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and device.type == "cuda")

print("Model loaded.")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"train_num_points: {model.config.train_num_points}")

Loading pretrained weights from facebook/mask2former-swin-small-ade-semantic ...


[transformers] Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-small-ade-semantic
Key                    | Status   |                                                                                           
-----------------------+----------+-------------------------------------------------------------------------------------------
class_predictor.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([151]) vs model:torch.Size([105])          
class_predictor.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([151, 256]) vs model:torch.Size([105, 256])
criterion.empty_weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([151]) vs model:torch.Size([105])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
u:\Programms\Python313\Lib\site-packages\torch\nn\modules\module.py:1367: UserWarning: expandable_segments not supported on this platform (Triggered

Gradient checkpointing unavailable: Mask2FormerForUniversalSegmentation does not support gradient checkpointing.
Model loaded.
Parameters: 68.7M
train_num_points: 4096


In [7]:
# Smoke test: one batch forward (no evaluate.load here — can hang on first Hub fetch)
batch = next(iter(train_dl))
with torch.amp.autocast("cuda", enabled=USE_AMP and device.type == "cuda"):
    outputs = model(
        pixel_values=batch["pixel_values"].to(device),
        mask_labels=[t.to(device) for t in batch["mask_labels"]],
        class_labels=[t.to(device) for t in batch["class_labels"]],
    )
print("Smoke test loss:", float(outputs.loss))
del outputs, batch
if device.type == "cuda":
    torch.cuda.empty_cache()

Smoke test loss: 13.323444366455078


C:\Users\Serge\AppData\Local\Temp\ipykernel_18656\3379063191.py:9: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  print("Smoke test loss:", float(outputs.loss))


In [8]:
import evaluate


def iou_per_class(pred: torch.Tensor, target: torch.Tensor, num_classes: int) -> list[float]:
    """pred, target: (N, H, W) integer class indices."""
    ious = []
    pred = pred.view(-1)
    target = target.view(-1)
    for c in range(num_classes):
        p = pred == c
        t = target == c
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        ious.append((inter / union).item() if union > 0 else float("nan"))
    return ious


def mean_iou_from_batches(batch_ious: list[list[float]]) -> float:
    if not batch_ious:
        return float("nan")
    arr = np.asarray(batch_ious, dtype=np.float64)
    per_class = np.nanmean(arr, axis=0)
    return float(np.nanmean(per_class))


def _batch_to_device(batch, device):
    return {
        "pixel_values": batch["pixel_values"].to(device, non_blocking=True),
        "mask_labels": [t.to(device, non_blocking=True) for t in batch["mask_labels"]],
        "class_labels": [t.to(device, non_blocking=True) for t in batch["class_labels"]],
    }


def _forward_loss(model, batch, device):
    inputs = _batch_to_device(batch, device)
    return model(**inputs)


def _predict_semantic_maps(model, batch, device, target_sizes=None):
    with torch.inference_mode():
        with torch.amp.autocast("cuda", enabled=USE_AMP and device.type == "cuda"):
            outputs = model(pixel_values=batch["pixel_values"].to(device, non_blocking=True))
        if target_sizes is None:
            target_sizes = [(img.shape[0], img.shape[1]) for img in batch["original_images"]]
        preds = processor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)
    return preds, outputs


def _targets_from_batch(batch, height: int, width: int) -> list[torch.Tensor]:
    targets = []
    for mask_labels, class_labels in zip(batch["mask_labels"], batch["class_labels"]):
        sem = torch.zeros((height, width), dtype=torch.long)
        for mask, cls_id in zip(mask_labels, class_labels):
            m = mask.bool()
            if m.shape != sem.shape:
                m = torch.nn.functional.interpolate(
                    m.unsqueeze(0).unsqueeze(0).float(),
                    size=(height, width),
                    mode="nearest",
                ).squeeze(0).squeeze(0).bool()
            sem[m] = cls_id.long()
        targets.append(sem)
    return targets


@torch.inference_mode()
def _compute_batch_miou(model, batch, device, target_size=None):
    """Fast mIoU at model input resolution for progress-bar display."""
    h, w = batch["pixel_values"].shape[-2:]
    if target_size is None:
        target_size = (h, w)
    target_sizes = [target_size] * batch["pixel_values"].size(0)
    preds, inf_outputs = _predict_semantic_maps(model, batch, device, target_sizes=target_sizes)
    del inf_outputs

    targets = _targets_from_batch(batch, target_size[0], target_size[1])
    pred_t = torch.stack([p.cpu() for p in preds]).long()
    tgt_t = torch.stack(targets).long()
    del preds, targets
    return iou_per_class(pred_t, tgt_t, num_classes)


def _progress_postfix(loss: float, miou: float | None = None) -> dict:
    postfix = {"loss": f"{loss:.4f}"}
    if miou is not None and not np.isnan(miou):
        postfix["mIoU"] = f"{miou:.4f}"
    return postfix


print("Loading evaluate mean_iou metric (first run may download from Hub)...")
miou_metric = evaluate.load("mean_iou")
print("Metric ready.")

Loading evaluate mean_iou metric (first run may download from Hub)...
Metric ready.


In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    device,
    grad_accum_steps=1,
    empty_cache_every=EMPTY_CACHE_EVERY,
    progress_iou_interval=PROGRESS_IOU_INTERVAL,
):
    model.train()
    running = 0.0
    n = 0
    batch_ious: list[list[float]] = []
    running_miou: float | None = None
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc="train", leave=False)
    for batch_idx, batch in enumerate(pbar):
        with torch.amp.autocast("cuda", enabled=USE_AMP and device.type == "cuda"):
            outputs = _forward_loss(model, batch, device)
            loss = outputs.loss / grad_accum_steps
        loss_value = float(outputs.loss.detach()) / grad_accum_steps
        del outputs

        if scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()
        del loss

        step_now = (batch_idx + 1) % grad_accum_steps == 0 or (batch_idx + 1) == len(loader)
        if step_now:
            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        if progress_iou_interval > 0 and (
            (batch_idx + 1) % progress_iou_interval == 0 or (batch_idx + 1) == len(loader)
        ):
            was_training = model.training
            model.eval()
            batch_ious.append(_compute_batch_miou(model, batch, device))
            if was_training:
                model.train()
            running_miou = mean_iou_from_batches(batch_ious)
            if device.type == "cuda":
                torch.cuda.empty_cache()

        bs = batch["pixel_values"].size(0)
        running += loss_value * grad_accum_steps * bs
        n += bs
        pbar.set_postfix(_progress_postfix(running / max(n, 1), running_miou))

        del batch
        if device.type == "cuda" and empty_cache_every > 0 and (batch_idx + 1) % empty_cache_every == 0:
            torch.cuda.empty_cache()

    epoch_miou = mean_iou_from_batches(batch_ious) if batch_ious else float("nan")
    return running / max(n, 1), epoch_miou


@torch.inference_mode()
def validate(
    model,
    loader,
    device,
    compute_miou=VAL_COMPUTE_MIOU,
    iou_interval=IOU_INTERVAL,
    progress_iou_interval=PROGRESS_IOU_INTERVAL,
):
    model.eval()
    val_loss = 0.0
    n = 0
    batch_ious: list[list[float]] = []
    progress_ious: list[list[float]] = []
    running_miou: float | None = None
    miou_metric_local = evaluate.load("mean_iou") if compute_miou else None

    pbar = tqdm(loader, desc="val", leave=False)
    for batch_idx, batch in enumerate(pbar):
        with torch.amp.autocast("cuda", enabled=USE_AMP and device.type == "cuda"):
            outputs = _forward_loss(model, batch, device)
        val_loss += float(outputs.loss) * batch["pixel_values"].size(0)
        n += batch["pixel_values"].size(0)
        del outputs

        should_compute_miou = compute_miou and (
            (batch_idx + 1) % iou_interval == 0 or (batch_idx + 1) == len(loader)
        )
        should_progress_miou = progress_iou_interval > 0 and (
            (batch_idx + 1) % progress_iou_interval == 0 or (batch_idx + 1) == len(loader)
        )

        if should_compute_miou or should_progress_miou:
            if should_compute_miou:
                preds, inf_outputs = _predict_semantic_maps(model, batch, device)
                del inf_outputs

                refs = [m.astype(np.int64) for m in batch["original_segmentation_maps"]]
                miou_metric_local.add_batch(
                    predictions=[p.cpu().numpy() for p in preds],
                    references=refs,
                )

                pred_t = torch.stack([p.cpu() for p in preds]).unsqueeze(1).float()
                tgt_t = torch.from_numpy(np.stack(refs)).unsqueeze(1).float()
                pred_rs = torch.nn.functional.interpolate(
                    pred_t, size=(IMAGE_SIZE, IMAGE_SIZE), mode="nearest"
                ).squeeze(1).long()
                tgt_rs = torch.nn.functional.interpolate(
                    tgt_t, size=(IMAGE_SIZE, IMAGE_SIZE), mode="nearest"
                ).squeeze(1).long()
                batch_ious.append(iou_per_class(pred_rs, tgt_rs, num_classes))
                del preds, pred_t, tgt_t, pred_rs, tgt_rs
            elif should_progress_miou:
                progress_ious.append(_compute_batch_miou(model, batch, device))
                running_miou = mean_iou_from_batches(progress_ious)

            if device.type == "cuda":
                torch.cuda.empty_cache()

        if should_compute_miou and batch_ious:
            running_miou = mean_iou_from_batches(batch_ious)

        pbar.set_postfix(_progress_postfix(val_loss / max(n, 1), running_miou))
        del batch

    avg_loss = val_loss / max(n, 1)
    if compute_miou and miou_metric_local is not None:
        hf_miou = miou_metric_local.compute(num_labels=num_classes, ignore_index=0)["mean_iou"]
        resized_miou = mean_iou_from_batches(batch_ious)
    elif progress_ious:
        hf_miou = float("nan")
        resized_miou = mean_iou_from_batches(progress_ious)
    else:
        hf_miou = float("nan")
        resized_miou = float("nan")
    return avg_loss, hf_miou, resized_miou


history = []
best_miou = -1.0
best_loss = float("inf")
best_ckpt = CHECKPOINT_DIR / "best"

for epoch in range(NUM_EPOCHS):
    print(f"{'-' * 20} Epoch {epoch + 1}/{NUM_EPOCHS} {'-' * 20}")
    train_loss, train_miou = train_one_epoch(
        model,
        train_dl,
        optimizer,
        scaler,
        device,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        empty_cache_every=EMPTY_CACHE_EVERY,
    )
    val_loss, val_miou, val_miou_resized = validate(model, test_dl, device)

    scheduler.step(val_loss)

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_miou": train_miou,
        "val_loss": val_loss,
        "val_miou": val_miou,
        "val_miou_resized": val_miou_resized,
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)

    if VAL_COMPUTE_MIOU:
        print(
            f"Train loss: {train_loss:.4f}, mIoU: {train_miou:.4f} | "
            f"Val loss: {val_loss:.4f} | Val mIoU (native): {val_miou:.4f} | "
            f"Val mIoU (resized): {val_miou_resized:.4f}"
        )
        improved = val_miou > best_miou
        if improved:
            best_miou = val_miou
    else:
        print(
            f"Train loss: {train_loss:.4f}, mIoU: {train_miou:.4f} | "
            f"Val loss: {val_loss:.4f}, mIoU: {val_miou_resized:.4f}"
        )
        improved = val_loss < best_loss
        if improved:
            best_loss = val_loss

    if improved:
        print(f"Saving best checkpoint to {best_ckpt}")
        model.save_pretrained(best_ckpt)
        processor.save_pretrained(best_ckpt)

    if device.type == "cuda":
        torch.cuda.empty_cache()

hist_df = pd.DataFrame(history)
hist_df

-------------------- Epoch 1/10 --------------------


train:   0%|          | 0/1246 [00:00<?, ?it/s]

C:\Users\Serge\AppData\Local\Temp\ipykernel_18656\1861731263.py:22: RuntimeWarning: Mean of empty slice
  per_class = np.nanmean(arr, axis=0)


val:   0%|          | 0/534 [00:00<?, ?it/s]

Train loss: 3.9570, mIoU: 0.0534 | Val loss: 3.0130, mIoU: 0.0751
Saving best checkpoint to u:\Dev\AI\HSS Project\checkpoints\mask2former\best
-------------------- Epoch 2/10 --------------------


...

Train loss: 1.9934, mIoU: 0.2501 | Val loss: 2.0575, mIoU: 0.1965
Saving best checkpoint to u:\Dev\AI\HSS Project\checkpoints\mask2former\best
-------------------- Epoch 8/10 --------------------


train:   0%|          | 0/1246 [00:00<?, ?it/s]

val:   0%|          | 0/534 [00:00<?, ?it/s]

Train loss: 1.8679, mIoU: 0.2411 | Val loss: 2.0663, mIoU: 0.2035
-------------------- Epoch 9/10 --------------------


train:   0%|          | 0/1246 [00:00<?, ?it/s]

In [ ]:
# Final evaluation on best checkpoint (full mIoU pass)
if best_ckpt.exists():
    final_model = Mask2FormerForUniversalSegmentation.from_pretrained(best_ckpt).to(device)
    patch_skip_auxiliary_loss(final_model)
else:
    final_model = model

final_loss, final_miou, final_miou_resized = validate(
    final_model,
    test_dl,
    device,
    compute_miou=True,
    iou_interval=max(1, IOU_INTERVAL // 5),
)
print(
    f"Final test | loss: {final_loss:.4f} | mIoU (native): {final_miou:.4f} | "
    f"mIoU (resized): {final_miou_resized:.4f}"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="Train loss")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"], marker="o", label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].set_title("Training / validation loss")

axes[1].plot(hist_df["epoch"], hist_df["train_miou"], marker="o", label="Train mIoU")
axes[1].plot(hist_df["epoch"], hist_df["val_miou_resized"], marker="s", label="Val mIoU (resized)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("mIoU")
axes[1].legend()
axes[1].set_title("Validation mIoU")
plt.tight_layout()
plt.show()

In [ ]:
def colorize_mask(mask: np.ndarray, train_id_to_color: Dict[int, Tuple[int, int, int]]) -> np.ndarray:
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for train_id in np.unique(mask):
        color = train_id_to_color.get(int(train_id), (255, 255, 255))
        color_mask[mask == train_id] = color
    return color_mask


def overlay_image_with_mask(image: np.ndarray, color_mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    image_f = image.astype(np.float32)
    mask_f = color_mask.astype(np.float32)
    return np.clip((1.0 - alpha) * image_f + alpha * mask_f, 0, 255).astype(np.uint8)


def show_gt_pred(image: np.ndarray, gt_mask: np.ndarray, pred_mask: np.ndarray, alpha: float = 0.45):
    gt_color = colorize_mask(gt_mask, train_id_to_color)
    pred_color = colorize_mask(pred_mask, train_id_to_color)
    gt_overlay = overlay_image_with_mask(image, gt_color, alpha=alpha)
    pred_overlay = overlay_image_with_mask(image, pred_color, alpha=alpha)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image)
    axes[0].set_title("Input")
    axes[0].axis("off")
    axes[1].imshow(gt_overlay)
    axes[1].set_title("Ground truth")
    axes[1].axis("off")
    axes[2].imshow(pred_overlay)
    axes[2].set_title("Prediction")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()


# Load best checkpoint for visualization
if best_ckpt.exists():
    viz_model = Mask2FormerForUniversalSegmentation.from_pretrained(best_ckpt).to(device)
else:
    viz_model = model
viz_model.eval()

rng = random.Random(SEED)
sample_indices = [rng.randrange(len(test_ds)) for _ in range(3)]

for idx in sample_indices:
    aug_img, aug_mask, orig_img, orig_mask = test_ds[idx]
    single_batch = collate_fn([(aug_img, aug_mask, orig_img, orig_mask)])
    preds, _ = _predict_semantic_maps(viz_model, single_batch, device)
    pred_mask = preds[0].cpu().numpy().astype(np.int64)
    print(f"idx={idx} | unique GT: {len(np.unique(orig_mask))} | unique pred: {len(np.unique(pred_mask))}")
    show_gt_pred(orig_img, orig_mask, pred_mask)

## Usage notes

- **`PYTORCH_CUDA_ALLOC_CONF`** must be set before `import torch`; restart kernel after changing.
- **OOM during training** (not instant): usually fragmentation or heavy batches. Defaults now use `TRAIN_NUM_POINTS=4096`, gradient checkpointing, `GRAD_ACCUM_STEPS=1`, and periodic `empty_cache()`. If still OOM: set `IMAGE_SIZE=384`, `TRAIN_NUM_POINTS=2048`, or `EMPTY_CACHE_EVERY=10`.
- **`SKIP_AUXILIARY_LOSS=True`**: skips auxiliary decoder losses (saves VRAM). Do **not** set `config.use_auxiliary_loss=False` — that breaks HF forward.
- **`PROGRESS_IOU_INTERVAL`**: mIoU shown in train/val tqdm bars every N batches (default 50). Set `0` to disable.
- **`VAL_COMPUTE_MIOU=False`**: epoch validation uses loss + progress-bar mIoU; run the final eval cell for full mIoU.
- **`BATCH_SIZE=1`**: raise `GRAD_ACCUM_STEPS` only after training is stable.
- **Resume training**: set `LOAD_CHECKPOINT = "checkpoints/mask2former/best"` in the config cell and re-run from model init onward.
- **mIoU comparison with `main.ipynb`**: use `val_miou_resized` from the final eval cell.